In [ ]:
import modal

image = (
    modal.Image.debian_slim(python_version="3.14")
    .uv_pip_install(
        "torch",
        "transformers",
        "trl",
        "peft",
        "accelerate",
        "datasets",
        "scikit-learn",
    )
)

app = modal.App("reward-model-ultrafeedback", image=image)

hf_secret = modal.Secret.from_name("huggingface")

In [ ]:
import torch
from datasets import load_dataset, DatasetDict
from transformers import AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig
from peft import LoraConfig

def build_dataset(train_size=20_000, val_size=1_000):
    split = load_dataset("trl-lib/ultrafeedback_binarized", split="train") \
        .shuffle(seed=42) \
        .train_test_split(test_size=0.1)

    ds = DatasetDict({
        "train": split["train"].select(range(train_size)),
        "val": split["test"].select(range(val_size)),
        "test": load_dataset("trl-lib/ultrafeedback_binarized", split="test"),
    })
    ds = ds.select_columns(["chosen", "rejected"])
    return ds

In [ ]:
REWARD_MODEL_REPO = "hyerra/qwen3-4b-reward-ultrafeedback"


@app.function(gpu="A100", secrets=[hf_secret], timeout=60 * 60 * 8)
def train_and_evaluate_reward():
    ds = build_dataset()

    model = AutoModelForSequenceClassification.from_pretrained(
        "Qwen/Qwen3-4B",
        num_labels=1,
        dtype=torch.bfloat16,
    )

    trainer = RewardTrainer(
        model,
        args=RewardConfig(
            output_dir="/tmp/reward-model",
            bf16=True,
            num_train_epochs=1,
            per_device_train_batch_size=16,
            eval_strategy="steps",
            save_strategy="steps",
            eval_steps=500,
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        ),
        train_dataset=ds["train"],
        eval_dataset=ds["val"],
        peft_config=LoraConfig(task_type="SEQ_CLS"),
    )

    trainer.train()

    trainer.model.eval()
    tokenizer = trainer.processing_class

    def score_batch(conversations, batch_size=32):
        scores = []
        for i in range(0, len(conversations), batch_size):
            batch = conversations[i:i + batch_size]
            enc = tokenizer.apply_chat_template(
                batch,
                tokenize=True,
                padding=True,
                add_generation_prompt=False,
                return_tensors="pt",
                return_dict=True,
            ).to(trainer.model.device)
            with torch.no_grad():
                out = trainer.model(**enc)
            scores.append(out.logits.squeeze(-1))
        return torch.cat(scores)

    chosen_scores = score_batch([s["chosen"] for s in ds["test"]])
    rejected_scores = score_batch([s["rejected"] for s in ds["test"]])
    accuracy = (chosen_scores > rejected_scores).float().mean().item()

    trainer.model.push_to_hub(REWARD_MODEL_REPO, private=True)
    tokenizer.push_to_hub(REWARD_MODEL_REPO, private=True)

    return accuracy

In [ ]:
with app.run():
    accuracy = train_and_evaluate_reward.remote()
print("Reward accuracy:", accuracy)